# Nykaa Domain Support Agent — Task 3: Chunking, Embeddings & ChromaDB

This notebook:
1. Loads the 12 policy documents from `knowledge_base/`.
2. Creates fixed-size chunks with overlap.
3. Creates sentence-based chunks.
4. Embeds both chunk sets using `all-MiniLM-L6-v2`.
5. Stores them in two separate persistent ChromaDB collections.
6. Runs a sample retrieval test against both collections.

**Run this notebook from the project folder that contains `knowledge_base/`.**

## 1. Install required packages

Run this cell once in your environment. If you are using a virtual environment, ensure the notebook kernel is connected to that same environment.

In [12]:
%pip install -U sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

We use `Path` to locate the policy files, `re` to split sentences, SentenceTransformers to create embeddings, and ChromaDB to store and search vectors.

In [13]:
from pathlib import Path
import re
import uuid

from sentence_transformers import SentenceTransformer
import chromadb

KNOWLEDGE_BASE_DIR = Path.cwd() / "knowledge_base"
CHROMA_DIR = Path.cwd() / "chroma_db"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

FIXED_CHUNK_SIZE = 300       # characters
FIXED_CHUNK_OVERLAP = 50     # characters
SENTENCES_PER_CHUNK = 2

print("Working directory:", Path.cwd())
print("Knowledge base directory:", KNOWLEDGE_BASE_DIR.resolve())
print("ChromaDB directory:", CHROMA_DIR.resolve())

if not KNOWLEDGE_BASE_DIR.exists():
    raise FileNotFoundError(
        "knowledge_base folder was not found. Run this notebook from the project folder "
        "that contains the knowledge_base folder."
    )


Working directory: c:\Users\singh\Downloads
Knowledge base directory: C:\Users\singh\Downloads\knowledge_base
ChromaDB directory: C:\Users\singh\Downloads\chroma_db


## 3. Load the policy documents

Each Markdown file becomes one source document. We retain its filename as `source`, because later evaluation must map retrieved chunks back to their parent documents.

In [14]:
def load_documents(folder):
    documents = []

    for file_path in sorted(folder.glob("*.md")):
        text = file_path.read_text(encoding="utf-8").strip()

        if text:
            documents.append({
                "source": file_path.name,
                "text": text
            })

    if not documents:
        raise ValueError("No non-empty Markdown documents were found.")

    return documents


documents = load_documents(KNOWLEDGE_BASE_DIR)

print(f"Loaded {len(documents)} documents.")
for document in documents:
    print(f"- {document['source']} ({len(document['text'])} characters)")

if len(documents) < 12:
    raise ValueError("The knowledge base must contain at least 12 documents.")


Loaded 12 documents.
- cancellation.md (438 characters)
- cod_refund.md (441 characters)
- damaged_item.md (513 characters)
- delivery_sla.md (500 characters)
- escalation_matrix.md (569 characters)
- international_shipping.md (409 characters)
- loyalty_points.md (489 characters)
- payment_failure.md (507 characters)
- return_window.md (633 characters)
- reverse_pickup.md (416 characters)
- size_exchange.md (432 characters)
- warranty.md (534 characters)


## 4. Strategy A — Fixed-size chunks with overlap

This method takes consecutive character windows. The next chunk starts before the previous chunk ends, so some text is shared between neighboring chunks.

With a chunk size of 300 and overlap of 50, the step size is 250 characters.

In [15]:
def fixed_size_chunk(text, chunk_size=FIXED_CHUNK_SIZE, overlap=FIXED_CHUNK_OVERLAP):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)

    return chunks


fixed_chunks = []

for document in documents:
    chunks = fixed_size_chunk(document["text"])

    for index, chunk_text in enumerate(chunks):
        fixed_chunks.append({
            "chunk_id": f"{document['source']}::fixed::{index}",
            "source": document["source"],
            "chunk_index": index,
            "text": chunk_text,
            "strategy": "fixed_size"
        })

print("Fixed-size chunks created:", len(fixed_chunks))
print("\nExample chunk:")
print(fixed_chunks[0]["text"])
print("\nParent document:", fixed_chunks[0]["source"])


Fixed-size chunks created: 29

Example chunk:
# Order-Cancellation Policy

A customer may request cancellation while an order is still in the Placed status and has not entered shipment processing. Once an order is Shipped, cancellation through the standard workflow is unavailable and the customer must follow the applicable return process after

Parent document: cancellation.md


## 5. Strategy B — Sentence-based chunks

This method first splits each document into sentences and then groups two consecutive sentences per chunk. It preserves sentence boundaries, although the final chunk may contain only one sentence.

In [16]:
def split_into_sentences(text):
    # The supplied policy documents use conventional sentence punctuation.
    sentences = re.split(r"(?<=[.!?])\\s+", text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def sentence_based_chunk(text, sentences_per_chunk=SENTENCES_PER_CHUNK):
    if sentences_per_chunk <= 0:
        raise ValueError("sentences_per_chunk must be greater than zero.")

    sentences = split_into_sentences(text)
    chunks = []

    for start in range(0, len(sentences), sentences_per_chunk):
        group = sentences[start:start + sentences_per_chunk]
        chunks.append(" ".join(group))

    return chunks


sentence_chunks = []

for document in documents:
    chunks = sentence_based_chunk(document["text"])

    for index, chunk_text in enumerate(chunks):
        sentence_chunks.append({
            "chunk_id": f"{document['source']}::sentence::{index}",
            "source": document["source"],
            "chunk_index": index,
            "text": chunk_text,
            "strategy": "sentence_based"
        })

print("Sentence-based chunks created:", len(sentence_chunks))
print("\nExample chunk:")
print(sentence_chunks[0]["text"])
print("\nParent document:", sentence_chunks[0]["source"])


Sentence-based chunks created: 12

Example chunk:
# Order-Cancellation Policy

A customer may request cancellation while an order is still in the Placed status and has not entered shipment processing. Once an order is Shipped, cancellation through the standard workflow is unavailable and the customer must follow the applicable return process after delivery. A cancellation request is not confirmed until the order record or support system shows that the status has changed to Cancelled.

Parent document: cancellation.md


## 6. Load the embedding model

The model converts each text chunk into a numerical vector. Similar meanings tend to have vectors that are closer in the embedding space.

The first run may download the model files, so it requires an internet connection once. After the model is available locally, it can be used offline.

In [17]:
embedding_model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\singh\AppData\Local\Temp\ipykernel_20860\2008840544.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


## 7. Generate embeddings for both chunking strategies

We embed the chunk text, not the filenames. The filenames are stored separately as metadata so that each retrieved chunk can be traced to its original policy document.

In [18]:
def create_embeddings(chunks, model, batch_size=32):
    texts = [chunk["text"] for chunk in chunks]

    vectors = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    return vectors.tolist()


fixed_embeddings = create_embeddings(fixed_chunks, embedding_model)
sentence_embeddings = create_embeddings(sentence_chunks, embedding_model)

print("Fixed embeddings:", len(fixed_embeddings))
print("Sentence embeddings:", len(sentence_embeddings))
print("Vector dimension:", len(fixed_embeddings[0]))


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Fixed embeddings: 29
Sentence embeddings: 12
Vector dimension: 384


## 8. Create two separate ChromaDB collections

Each strategy gets its own collection. We use persistent storage so the vectors remain available after the notebook kernel stops.

If you rerun this notebook, the existing collections are deleted and rebuilt from the current policy files and settings. This prevents duplicate records from accumulating.

In [19]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

COLLECTION_FIXED = "nykaa_fixed_chunks"
COLLECTION_SENTENCE = "nykaa_sentence_chunks"

# Delete existing collections before rebuilding
for collection_name in [
    "nykaa_fixed_chunks",
    "nykaa_sentence_chunks"
]:
    try:
        client.delete_collection(name=collection_name)
    except Exception:
        pass


# Create collections using cosine distance
fixed_collection = client.create_collection(
    name="nykaa_fixed_chunks",
    metadata={"hnsw:space": "cosine"}
)

sentence_collection = client.create_collection(
    name="nykaa_sentence_chunks",
    metadata={"hnsw:space": "cosine"}
)

print("Both collections created using cosine distance.")


Both collections created using cosine distance.


## 9. Store chunks, embeddings and metadata

Each ChromaDB record stores:
- `id`: unique chunk identifier.
- `embedding`: numerical vector.
- `document`: the original chunk text.
- `metadata`: parent filename, chunk index and chunking strategy.

In [20]:
def add_chunks_to_collection(collection, chunks, embeddings, batch_size=100):
    if not (len(chunks) == len(embeddings)):
        raise ValueError("Number of chunks and embeddings must match.")

    for start in range(0, len(chunks), batch_size):
        end = start + batch_size
        batch_chunks = chunks[start:end]
        batch_embeddings = embeddings[start:end]

        collection.add(
            ids=[chunk["chunk_id"] for chunk in batch_chunks],
            documents=[chunk["text"] for chunk in batch_chunks],
            embeddings=batch_embeddings,
            metadatas=[
                {
                    "source": chunk["source"],
                    "chunk_index": chunk["chunk_index"],
                    "strategy": chunk["strategy"]
                }
                for chunk in batch_chunks
            ]
        )


add_chunks_to_collection(fixed_collection, fixed_chunks, fixed_embeddings)
add_chunks_to_collection(sentence_collection, sentence_chunks, sentence_embeddings)

print("Fixed collection records:", fixed_collection.count())
print("Sentence collection records:", sentence_collection.count())


Fixed collection records: 29
Sentence collection records: 12


## 10. Test retrieval from both collections

Use the same sample question against both collections. ChromaDB returns the nearest chunks according to vector distance. Because the embeddings are normalized, cosine distance can be calculated as `1 - cosine similarity`.

In [21]:
def search_collection(collection, query, model, top_k=3):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for text, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        similarity = 1 - distance

        retrieved.append({
            "text": text,
            "source": metadata["source"],
            "similarity": similarity
        })

    return retrieved


sample_query = "How long does a cash-on-delivery refund take?"

print("QUERY:", sample_query)

for label, collection in [
    ("FIXED-SIZE COLLECTION", fixed_collection),
    ("SENTENCE-BASED COLLECTION", sentence_collection)
]:
    print("\n" + "=" * 65)
    print(label)
    print("=" * 65)

    results = search_collection(collection, sample_query, embedding_model, top_k=3)

    for rank, result in enumerate(results, start=1):
        print(f"\nRank {rank}")
        print("Source:", result["source"])
        print(f"Cosine similarity: {result['similarity']:.4f}")
        print("Text:", result["text"])


QUERY: How long does a cash-on-delivery refund take?

FIXED-SIZE COLLECTION

Rank 1
Source: cod_refund.md
Cosine similarity: 0.6782
Text: # Cash-on-Delivery (COD) Refund Timelines

For this simulated support environment, an approved COD refund is issued through the bank-account or UPI details collected during the refund process. The expected processing time is 5–7 business days after the returned item passes the required inspection. I

Rank 2
Source: cod_refund.md
Cosine similarity: 0.5095
Text: he returned item passes the required inspection. If the refund has not arrived after 7 business days, the customer should contact support with the order ID and refund reference, if available.

Rank 3
Source: delivery_sla.md
Cosine similarity: 0.4062
Text: # Delivery Service-Level Agreements

For this synthetic project, standard domestic orders have an estimated delivery window of 3–7 business days from shipment confirmation. Remote delivery locations may require up to 10 business days, and the 

## 11. Final Task 3 checks

Confirm that both collections contain records and that the chunks retain their parent document metadata.

In [22]:
assert fixed_collection.count() == len(fixed_chunks)
assert sentence_collection.count() == len(sentence_chunks)

# Inspect one stored record from each collection.
print("Fixed collection sample:")
print(fixed_collection.peek(1))

print("\nSentence collection sample:")
print(sentence_collection.peek(1))

print("\nPASS: Both chunking strategies were embedded and indexed.")
print("PASS: Both ChromaDB collections contain their expected records.")
print("PASS: Parent document metadata is stored with each chunk.")


Fixed collection sample:
{'ids': ['cancellation.md::fixed::0'], 'embeddings': array([[-2.29276679e-02,  3.62496711e-02,  5.53100109e-02,
         4.06583995e-02, -1.91595424e-02, -4.02145274e-02,
         5.19659184e-02, -1.24607682e-01,  1.24777723e-02,
         4.04623635e-02,  8.47173780e-02,  1.16160989e-01,
         7.75063236e-04, -8.02520216e-02, -3.64738554e-02,
        -2.80380007e-02, -1.82166453e-02, -4.30569984e-02,
        -1.01164885e-01,  6.10649213e-03,  3.03921737e-02,
         5.76564670e-02, -5.15417717e-02,  7.20482841e-02,
        -1.03360182e-02,  4.04384844e-02, -8.05874988e-02,
         3.78336273e-02, -6.77373540e-03,  4.95724827e-02,
        -2.47965436e-02,  2.88183689e-02,  3.04492414e-02,
        -1.31402267e-02, -1.11229224e-02, -3.11402902e-02,
        -4.81235087e-02, -8.46363977e-02, -3.36735323e-02,
        -4.20448259e-02, -3.45556601e-03, -4.48858086e-03,
        -9.34919193e-02, -1.08049251e-02,  2.67069582e-02,
        -1.41806726e-03, -4.57006320e